In [1]:
import numpy as np
import matplotlib.pyplot as plt

dat=np.load('/home/maria/QuantumCryptography/InfoTheoryForBCI/001/1sec_first_data.npy')
print(dat.shape)

(1250, 83)


In [2]:
import numpy as np
import matplotlib.pyplot as plt

# dat already has shape (1250, 83)
x = dat.astype(np.float64)

# Flatten all timepoints and channels to learn one global quantizer
x_flat = x.ravel()

print("Shape:", x.shape)
print("Number of samples:", x_flat.size)
print("Minimum:", x_flat.min())
print("Maximum:", x_flat.max())
print("Mean:", x_flat.mean())
print("Standard deviation:", x_flat.std())

Shape: (1250, 83)
Number of samples: 103750
Minimum: -0.0006807449972257018
Maximum: 0.0004178849922027439
Mean: -3.191421423710392e-08
Standard deviation: 9.948683757126013e-05


In [3]:
def symbol_statistics(symbols, n_symbols):
    """
    Calculate counts, empirical probabilities, and self-information.
    """
    flat = np.asarray(symbols).ravel()

    counts = np.bincount(flat, minlength=n_symbols)
    probabilities = counts / counts.sum()

    surprise = np.full(n_symbols, np.inf, dtype=float)
    observed = probabilities > 0
    surprise[observed] = -np.log2(probabilities[observed])

    return counts, probabilities, surprise


def reconstruction_metrics(original, reconstructed):
    """
    Measure the distortion introduced by quantization.
    """
    original = np.asarray(original, dtype=float)
    reconstructed = np.asarray(reconstructed, dtype=float)

    mse = np.mean((original - reconstructed) ** 2)
    variance = np.var(original)

    normalized_mse = mse / variance if variance > 0 else np.nan

    signal_power = np.mean(original ** 2)
    sqnr_db = (
        10 * np.log10(signal_power / mse)
        if mse > 0 and signal_power > 0
        else np.inf
    )

    return {
        "mse": mse,
        "normalized_mse": normalized_mse,
        "sqnr_db": sqnr_db,
    }


def print_quantizer_summary(name, symbols, reconstruction, n_symbols):
    counts, probabilities, surprise = symbol_statistics(
        symbols, n_symbols
    )
    metrics = reconstruction_metrics(x, reconstruction)

    entropy = -np.sum(
        probabilities[probabilities > 0]
        * np.log2(probabilities[probabilities > 0])
    )

    print(f"\n{name}")
    print("-" * len(name))
    print("Counts:", counts)
    print("Probabilities:", np.round(probabilities, 4))
    print("Surprise in bits:", np.round(surprise, 3))
    print(f"Entropy: {entropy:.4f} bits/symbol")
    print(f"MSE: {metrics['mse']:.6g}")
    print(f"Normalized MSE: {metrics['normalized_mse']:.6g}")
    print(f"SQNR: {metrics['sqnr_db']:.2f} dB")

def uniform_quantize(x, n_bins=4):
    x = np.asarray(x, dtype=float)

    edges = np.linspace(x.min(), x.max(), n_bins + 1)

    # digitize returns labels from 0 to n_bins - 1
    symbols = np.digitize(x, edges[1:-1])

    # Reconstruct each symbol using the midpoint of its interval
    centers = (edges[:-1] + edges[1:]) / 2
    reconstruction = centers[symbols]

    return symbols, reconstruction, edges, centers


uniform_symbols, uniform_recon, uniform_edges, uniform_centers = (
    uniform_quantize(x, n_bins=4)
)

print_quantizer_summary(
    "Uniform-width quantizer",
    uniform_symbols,
    uniform_recon,
    n_symbols=4,
)

print("Edges:", uniform_edges)
print("Centers:", uniform_centers)


Uniform-width quantizer
-----------------------
Counts: [  264  8304 88012  7170]
Probabilities: [0.0025 0.08   0.8483 0.0691]
Surprise in bits: [8.618 3.643 0.237 3.855]
Entropy: 0.7813 bits/symbol
MSE: 4.60943e-09
Normalized MSE: 0.46571
SQNR: 3.32 dB
Edges: [-0.00068074 -0.00040609 -0.00013143  0.00014323  0.00041788]
Centers: [-5.43416249e-04 -2.68758751e-04  5.89874617e-06  2.80556244e-04]
